# **CNN yordamida tasvirlarni tasniflash**

## Ushbu noutbukda

### MNIST ma'lumotlar to'plami yordamida

### qo'lda yozilgan raqamlarni

### tasniflash uchun

### Birlashtirilgan Neyron Tarmoq (CNN)

### yaratilishi va shug'ullantirishni o'rganamiz

# **1. Ma’lumotlarni tayyorlash**

In [85]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms

In [86]:
torch.cuda.is_available()


True

In [87]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [88]:
data_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

In [89]:
# FashionMNIST
train_date = torchvision.datasets.FashionMNIST(
     root='./data',
     train=True,
     download=True,
     transform=data_transforms
)

test_date = torchvision.datasets.FashionMNIST(
     root='./data',
     train=False,
     download=True,
     transform=data_transforms
)

In [90]:
train_loader = torch.utils.data.DataLoader(
    train_date,
    batch_size=32,
    shuffle=True
)

test_loader = torch.utils.data.DataLoader(
    test_date,
    batch_size=32,
    shuffle=False
)

In [91]:
misol = next(iter(train_loader))
misol[0].shape

torch.Size([32, 1, 28, 28])

### Avval FashionMNIST ma'lumotlar to'plamini yuklab oldik. Bu to'plam 28x28 pikselli, 1 kanalli (kulrang tusli) kiyim tasvirlaridan iborat.
### Tasvirlarni model uchun mos formatga keltirish maqsadida ularni ToTensor() yordamida tenzorlarga o'tkazdik va Normalize() yordamida normalizatsiya qildik.
### Ma'lumotlarni o'quv (train) va sinov (test) to'plamlariga ajratib, ularni kichik batch larga bo'lib yuklash uchun DataLoader'lardan foydalandik.

# **2. CNN Modelini qurish**

In [94]:
class KiyimCNN(nn.Module):  # KiyimCNN klassi oddiy Konvolyutsion Neyron Tarmoq (CNN) modelini aniqlaydi:
    def __init__(self):
        super(KiyimCNN, self).__init__()

        self.net = nn.Sequential(
            # 2 blocks with Covn, Relu, Maxpool
            nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(in_channels=8, out_channels=32, kernel_size=3, padding=1), # Corrected in_channels to 8
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Flatten, Linear, relu, linear
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
      return self.net(x)

In [97]:
model = KiyimCNN().to(device)
print(model)

KiyimCNN(
  (net): Sequential(
    (0): Conv2d(1, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(8, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Flatten(start_dim=1, end_dim=-1)
    (7): Linear(in_features=1568, out_features=128, bias=True)
    (8): ReLU()
    (9): Linear(in_features=128, out_features=10, bias=True)
  )
)


### KiyimCNN nomli konvolyutsion neyron tarmoqni yaratdik. Bu tarmoq ikki qatlamli konvolyutsion bloklardan (Conv2d, ReLU, MaxPool2d) iborat bo'lib, ular tasvirlardagi muhim xususiyatlarni aniqlashga yordam beradi.
### So'ngra, topilgan xususiyatlar asosida tasvirlarni 10 xil kiyim turidan biriga tasniflash uchun Flatten va Linear (to'liq bog'langan) qatlamlar ishlatildi.
### Modelni hisoblash uchun tezroq bo'lishi uchun mavjud bo'lsa GPU (aks holda CPU) ga o'tkazdik.

# **3. Modelni shug‘ullantirish va baholash**

In [101]:
net_model = KiyimCNN()


In [102]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net_model.parameters(), lr=0.002)

In [103]:
epoxa = 5
# Model 5 marta to'liq ma'lumotlar to'plami bo'yicha o'qitiladi (har bir to'liq o'tish "epoxa" deyiladi).

net_model.to(device) # Modelni GPUga (agar mavjud bo'lsa) yoki CPUga o'tkazadi, bu hisoblashlarni tezlashtiradi.

for epox in range(epoxa):
  net_model.train()
  total_loss = 0
  for data, target in train_loader:
    data, target = data.to(device), target.to(device)

    # Forward - Oldga yuborish
    output = net_model(data)
    loss = criterion(output, target)

    # backward - Ortga yuborish
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    total_loss += loss.item()

  print(f"Epoxa: {epox+1}, O'rtacha yo'qotish: {total_loss/len(train_loader)}")

Epoxa: 1, O'rtacha yo'qotish: 0.4139801691770554
Epoxa: 2, O'rtacha yo'qotish: 0.2738734613756339
Epoxa: 3, O'rtacha yo'qotish: 0.2290366165647904
Epoxa: 4, O'rtacha yo'qotish: 0.1975007280121247
Epoxa: 5, O'rtacha yo'qotish: 0.17178456186999877


In [104]:
net_model.eval()

correct = 0
total = 0

with torch.no_grad():
  for data, target in test_loader:
    data, target = data.to(device), target.to(device)
    output = net_model(data)

    _, predicted = torch.max(output.data, 1)
    total += target.size(0)
    correct += (predicted == target).sum().item()

print(f"Test accuracy: {100 * correct / total}%")

Test accuracy: 91.25%


### Modelni shug'ullantirish uchun nn.CrossEntropyLoss yo'qotish funksiyasini (modelning xatosini o'lchash uchun) va optim.Adam optimizatorini (model parametrlarini yangilash uchun) tanladik. O'rganish darajasi (lr) 0.002 ga o'rnatildi.
### Modelni 5 marta to'liq ma'lumotlar to'plami bo'yicha shug'ullantirdik (ya'ni 5 epoxa davomida). Har bir epoxa oxirida modelning o'rtacha yo'qotishi ekranga chiqarildi, bu modelning qanday o'rganayotganini ko'rsatdi.
### Shug'ullantirish tugagandan so'ng, modelning sinov ma'lumotlari to'plamidagi aniqligini (accuracy) hisoblab, uni foizda chop etdik.

### **Nima sodir bo'ldi va nima bashorat qilindi:**

### Model 5 epoxa davomida FashionMNIST ma'lumotlar to'plamidagi turli kiyim tasvirlarini (masalan, futbolka, shim, krossovka kabi 10 ta tur) tanib olishni o'rgandi.
### Shug'ullantirish jarayonida modelning yo'qotishi (xatosi) doimiy ravishda kamayib bordi, bu modelning tasvirlardagi naqshlarni samarali o'zlashtirganini ko'rsatadi.
### Yakuniy baholash natijasida, model sinov ma'lumotlari to'plamidagi (model avval ko'rmagan) tasvirlarni tasniflashda 91.25% aniqlikka erishdi. Bu shuni anglatadiki, model har 100 ta tasvirdan taxminan 91 tasini to'g'ri aniqladi.


## **Yana ham aniqroq aytadigan bo'lsak:**

 Model yakuniy qatlamda 0 dan 9 gacha bo'lgan 10 ta raqamdan birini bashorat qiladi. Har bir raqam ma'lum bir kiyim turiga mos keladi. Misol uchun:

0 - T-shirt/top (futbolka/mayka)

1 - Trouser (shim)

2 - Pullover (sviter)

3 - Dress (ko'ylak)


4 - Coat (palto)

5 - Sandal (sandal)

6 - Shirt (ko'ylak)

7 - Sneaker (krossovka)

8 - Bag (sumka)

9 - Ankle boot (botinka)

Shunday qilib, model 7 raqamini bashorat qilsa, bu u krossovka deb aniqlaganini bildiradi.

